In [1]:
import numpy as np

In [ ]:
def householder_qr_simple(A_init, b_init, eps=1e-12):
    A = A_init.copy().astype(float)
    b = b_init.copy().astype(float)
    n = len(A)
    Q = np.eye(n)

    # Q = I; Lr
    for r in range(n - 1):
        # Construct matrix Pr, constant β, and vector u
        # σ = Σ aᵢ²
        sigma = 0.0
        for i in range(r, n):
            sigma += A[i, r] ** 2

        if np.sqrt(sigma) <= eps:
            print(f"Matrix nearly singular at column {r}")
            break
      
        # k = -sgmn(aᵣ)√σ; sgmn(x) = +1 if x≥0, -1 if x<0
        if A[r, r] >= 0:
            sign_ar = 1.0 
        else: 
            sign_ar = -1.0

        k = -sign_ar * np.sqrt(sigma)

        # β = σ - k·aᵣ
        beta = sigma - k * A[r, r]
        
        if abs(beta) < eps:
            continue

        # uᵣ = aᵣ - k; uᵢ = aᵣᵢ for i = r+1,...,n
        u = np.zeros(n)
        u[r] = A[r, r] - k
        for i in range(r + 1, n):
            u[i] = A[i, r]

        # A = Pr * A
        # Transform columns j = r,...,n-1
        for j in range(r, n):
            # γ = (Aₑⱼ·u)/β = (Σ aᵢⱼ·uᵢ)/β
            gamma = 0.0
            for i in range(r, n):
                gamma += u[i] * A[i, j]
            gamma /= beta
            # aᵢⱼ = aᵢⱼ - γ·uᵢ
            for i in range(r, n):
                A[i, j] -= gamma * u[i]

        # b = Pr * b
        # aᵣᵣ = k; aᵢᵣ = 0, i = r+1,...,n
        # γ = (b·u)/β = (Σ bᵢ·uᵢ)/β
        gamma_b = 0.0
        for i in range(r, n):
            gamma_b += u[i] * b[i]
        gamma_b /= beta
        # bᵢ = bᵢ - γ·uᵢ
        for i in range(r, n):
            b[i] -= gamma_b * u[i]

        # Q = Pr * Q
        # For j = 1,...,n
        for j in range(n):
            # γ = (Qₑⱼ·u)/β = (Σ qᵢⱼ·uᵢ)/β
            gamma_Q = 0.0
            for i in range(r, n):
                gamma_Q += u[i] * Q[i, j]
            gamma_Q /= beta
            # qᵢⱼ = qᵢⱼ - γ·uᵢ
            for i in range(r, n):
                Q[i, j] -= gamma_Q * u[i]

        A[r, r] = k
        for i in range(r + 1, n):
            A[i, r] = 0.0

    
    return Q.T, A, b


In [3]:
def back_substitution(R, y, eps=1e-12):
    n = len(y)
    x = np.zeros(n)
    for i in reversed(range(n)):
        s = 0.0
        for j in range(i + 1, n):
            s += R[i, j] * x[j]

        if abs(R[i, i]) < eps:
            raise ValueError("Singular or ill-conditioned matrix during back substitution.")
        
        x[i] = (y[i] - s) / R[i, i]
    return x

In [ ]:
def inverse_householder(A, Q, R, eps=1e-12):
    n = len(A)
    
    for i in range(n):
        if abs(R[i, i]) < eps:
            raise ValueError(f"Matrix A is singular or ill-conditioned (|R[{i},{i}]| = {abs(R[i, i])} < {eps}). Inverse cannot be calculated.")
    
    A_inv = np.zeros((n, n))
    

    for j in range(n):
        e = np.zeros(n)
        e[j] = 1.0
        b = Q.T @ e
        
        x = back_substitution(R, b, eps)
        
        for i in range(n):
            A_inv[i, j] = x[i]
    
    return A_inv

In [6]:

np.set_printoptions(precision=6, suppress=True)
  
n = int(input("Enter matrix dimension n: ") or 4)
eps = 1e-8
np.random.seed(42)  

A = np.random.rand(n, n)
s = np.random.rand(n)
b = np.zeros(n)
for i in range(n):  
    for j in range(n):
        b[i] += A[i, j] * s[j]

print("\nMatrix A:\n", A)
print("Vector s:", s)
print("Vector b = A*s:", b)

   
Q, R, b_trans = householder_qr_simple(A, b, eps)
x_H = back_substitution(R, Q.T @ b)
Q_lib, R_lib = np.linalg.qr(A)
x_QR = np.linalg.solve(R_lib, Q_lib.T @ b)

err_x_diff = np.linalg.norm(x_QR - x_H, 2)
err_Axb_H = np.linalg.norm(A @ x_H - b, 2)
err_Axb_QR = np.linalg.norm(A @ x_QR - b, 2)
err_xs_H = np.linalg.norm(x_H - s, 2) / np.linalg.norm(s, 2)
err_xs_QR = np.linalg.norm(x_QR - s, 2) / np.linalg.norm(s, 2)


A_inv_H = inverse_householder(A, Q, R)
A_inv_lib = np.linalg.inv(A)
err_inv = np.linalg.norm(A_inv_H - A_inv_lib, 2)


print("\n--- Results ---")
print("x_Householder =", x_H)
print("x_QR (library) =", x_QR)
print("||x_QR - x_H||2 =", err_x_diff)
print("\nErrors:")
print("||A x_H - b||2 =", err_Axb_H)
print("||A x_QR - b||2 =", err_Axb_QR)
print("||x_H - s|| / ||s|| =", err_xs_H)
print("||x_QR - s|| / ||s|| =", err_xs_QR)
print("\nInverse comparison:")
print("||A^-1_H - A^-1_lib||2 =", err_inv)





Matrix A:
 [[0.37454  0.950714 0.731994 ... 0.427541 0.025419 0.107891]
 [0.031429 0.63641  0.314356 ... 0.89711  0.887086 0.779876]
 [0.642032 0.08414  0.161629 ... 0.215821 0.62289  0.085347]
 ...
 [0.780733 0.215005 0.715636 ... 0.645346 0.414736 0.930351]
 [0.784883 0.200009 0.130842 ... 0.679818 0.830922 0.041596]
 [0.023111 0.373007 0.148788 ... 0.946708 0.397488 0.21714 ]]
Vector s: [0.373641 0.332912 0.176154 0.607267 0.476624 0.865701 0.03211  0.643868
 0.762949 0.759487 0.886074 0.729034 0.92781  0.332657 0.503209 0.01408
 0.006958 0.240127 0.100807 0.260211 0.177043 0.02852  0.909304 0.008223
 0.736082 0.152148 0.91223  0.892796 0.653901 0.672234 0.005339 0.984872
 0.877833 0.895106 0.659245 0.344787 0.893649 0.291517 0.562712 0.099731
 0.921956 0.14077  0.224897 0.558134 0.764093 0.619351 0.232134 0.000943
 0.757151 0.985207 0.809913 0.460207 0.903767 0.409848 0.524934 0.47713
 0.447624 0.775191 0.526769 0.316717 0.258158 0.037747 0.583195 0.229961
 0.148134 0.65383  0.681